# NMS Questionnaire Data Cleaning and Feature Preparation

###  Parkinson's Disease Screening Project

This notebook performs:
- Loading the questionnaire dataset
- Data inspection
- Missing value and duplicate checks
- Response validation
- Feature engineering
- Questionnaire summary generation
- Saving the cleaned dataset

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path


# Loading Dataset

In [4]:
# File locations
input_file = Path("../data/interim/questionnaire.csv")
output_folder = Path("../data/processed")

# Creating the processed folder if it does not exist
output_folder.mkdir(parents=True, exist_ok=True)

# Checking whether the file exists
print("File exists:", input_file.exists())

# Loading the dataset
questionnaire = pd.read_csv(input_file)

print("Dataset loaded successfully.")
print("Number of rows:", questionnaire.shape[0])
print("Number of columns:", questionnaire.shape[1])

questionnaire.head()

File exists: True
Dataset loaded successfully.
Number of rows: 469
Number of columns: 32


,patient_id,questionnaire_name,Q01,Q02,Q03,Q04,Q05,Q06,Q07,Q08,...,Q21,Q22,Q23,Q24,Q25,Q26,Q27,Q28,Q29,Q30
0,1,NMS,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,NMS,1,1,0,0,1,0,0,0,...,1,1,1,0,1,0,1,0,1,0
2,3,NMS,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,NMS,0,1,0,1,0,0,0,1,...,1,1,1,0,0,1,1,1,0,0
4,5,NMS,1,1,1,0,0,0,0,1,...,0,1,1,1,1,1,1,0,0,0


# Creating working copy 

In [5]:
# Keeping the originally loaded dataset unchanged
questionnaire_clean = questionnaire.copy()

print("A working copy of the dataset was created.")

A working copy of the dataset was created.


# Examining the dataset 

In [6]:
print("Dataset shape:", questionnaire_clean.shape)

print("\nColumn names:")
print(questionnaire_clean.columns.tolist())

print("\nDataset information:")
questionnaire_clean.info()

Dataset shape: (469, 32)

Column names:
['patient_id', 'questionnaire_name', 'Q01', 'Q02', 'Q03', 'Q04', 'Q05', 'Q06', 'Q07', 'Q08', 'Q09', 'Q10', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19', 'Q20', 'Q21', 'Q22', 'Q23', 'Q24', 'Q25', 'Q26', 'Q27', 'Q28', 'Q29', 'Q30']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 469 entries, 0 to 468
Data columns (total 32 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   patient_id          469 non-null    int64 
 1   questionnaire_name  469 non-null    object
 2   Q01                 469 non-null    int64 
 3   Q02                 469 non-null    int64 
 4   Q03                 469 non-null    int64 
 5   Q04                 469 non-null    int64 
 6   Q05                 469 non-null    int64 
 7   Q06                 469 non-null    int64 
 8   Q07                 469 non-null    int64 
 9   Q08                 469 non-null    int64 
 10  Q09  

# Checking questionnaire name 

In [7]:
questionnaire_clean["questionnaire_name"].value_counts(dropna=False)

questionnaire_name
NMS    469
Name: count, dtype: int64

# Identifying Q01 to Q30

In [8]:
# Creating a list containing Q01, Q02, ..., Q30
question_cols = [f"Q{i:02d}" for i in range(1, 31)]

print("Number of questionnaire questions:", len(question_cols))
print(question_cols)

Number of questionnaire questions: 30
['Q01', 'Q02', 'Q03', 'Q04', 'Q05', 'Q06', 'Q07', 'Q08', 'Q09', 'Q10', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19', 'Q20', 'Q21', 'Q22', 'Q23', 'Q24', 'Q25', 'Q26', 'Q27', 'Q28', 'Q29', 'Q30']


In [9]:
missing_columns = [
    col for col in question_cols
    if col not in questionnaire_clean.columns
]

if len(missing_columns) == 0:
    print("All Q01 to Q30 columns are available.")
else:
    print("Missing columns:", missing_columns)

All Q01 to Q30 columns are available.


# Checking for missing values and duplicates in the dataset

In [10]:
print(
    "Missing patient IDs:",
    questionnaire_clean["patient_id"].isna().sum()
)

print(
    "Missing questionnaire responses:",
    questionnaire_clean[question_cols].isna().sum().sum()
)

print(
    "Exact duplicate rows:",
    questionnaire_clean.duplicated().sum()
)

print(
    "Duplicate patient IDs:",
    questionnaire_clean["patient_id"].duplicated().sum()
)

Missing patient IDs: 0
Missing questionnaire responses: 0
Exact duplicate rows: 0
Duplicate patient IDs: 0


# Converting questionnaire responses to numeric 

In [11]:
questionnaire_clean[question_cols] = questionnaire_clean[
    question_cols
].apply(
    pd.to_numeric,
    errors="coerce"
)

print("Q01 to Q30 were converted to numeric format.")

Q01 to Q30 were converted to numeric format.


# Checking valid and invalid responses 


In [12]:
# Checking for invalid questionnaire responses
invalid_count = 0

for col in question_cols:
    invalid_condition = (
        questionnaire_clean[col].notna()
        & ~questionnaire_clean[col].isin([0, 1])
    )

    invalid_count += invalid_condition.sum()

    # Replace invalid values only when they exist
    questionnaire_clean.loc[invalid_condition, col] = np.nan

if invalid_count == 0:
    print("No invalid values were found. No replacement was needed.")
else:
    print(
        invalid_count,
        "invalid responses were replaced with missing values."
    )

No invalid values were found. No replacement was needed.


# Removing exact duplicate rows

In [13]:
# Counting the number of rows before removing duplicates
rows_before = len(questionnaire_clean)

# Removing only completely identical rows
questionnaire_clean = (
    questionnaire_clean
    .drop_duplicates()
    .reset_index(drop=True)
)

# Counting the number of rows after removing duplicates
rows_after = len(questionnaire_clean)

print("Rows before removing duplicates:", rows_before)
print("Rows after removing duplicates:", rows_after)
print("Duplicate rows removed:", rows_before - rows_after)

Rows before removing duplicates: 469
Rows after removing duplicates: 469
Duplicate rows removed: 0


# Creating questionnaire completion variables 

In [14]:
# Counting how many questions each participant answered
questionnaire_clean["questions_answered"] = (
    questionnaire_clean[question_cols]
    .notna()
    .sum(axis=1)
)

# Counting how many questions are missing for each participant
questionnaire_clean["questions_missing"] = (
    questionnaire_clean[question_cols]
    .isna()
    .sum(axis=1)
)

# Identifying whether each questionnaire is complete
questionnaire_clean["questionnaire_status"] = np.where(
    questionnaire_clean["questions_missing"] == 0,
    "Complete",
    "Incomplete"
)

# Displaying the new columns
questionnaire_clean[
    [
        "patient_id",
        "questions_answered",
        "questions_missing",
        "questionnaire_status"
    ]
].head()

,patient_id,questions_answered,questions_missing,questionnaire_status
0,1,30,0,Complete
1,2,30,0,Complete
2,3,30,0,Complete
3,4,30,0,Complete
4,5,30,0,Complete


# Checking questionnaire completion 

In [15]:
# Counting complete and incomplete questionnaires
if "questionnaire_status" not in questionnaire_clean.columns:
    questionnaire_clean["questions_answered"] = questionnaire_clean[question_cols].notna().sum(axis=1)
    questionnaire_clean["questions_missing"] = questionnaire_clean[question_cols].isna().sum(axis=1)
    questionnaire_clean["questionnaire_status"] = np.where(
        questionnaire_clean["questions_missing"] == 0,
        "Complete",
        "Incomplete"
    )

questionnaire_clean["questionnaire_status"].value_counts(dropna=False)

questionnaire_status
Complete    469
Name: count, dtype: int64

# Creating the total symptom count

In [16]:
# Adding the responses from Q01 to Q30
# A response of 1 represents a reported symptom
questionnaire_clean["total_symptom_count"] = (
    questionnaire_clean[question_cols]
    .sum(axis=1, min_count=1)
)

# Displaying the result
questionnaire_clean[
    [
        "patient_id",
        "questionnaire_name",
        "total_symptom_count"
    ]
].head(10)

,patient_id,questionnaire_name,total_symptom_count
0,1,NMS,0.0
1,2,NMS,12.0
2,3,NMS,0.0
3,4,NMS,12.0
4,5,NMS,11.0
5,6,NMS,16.0
6,7,NMS,16.0
7,8,NMS,12.0
8,9,NMS,2.0
9,10,NMS,6.0


# Domain level count 

In [17]:
# Official PADS symptom categories
symptom_categories = {
    "gastrointestinal_count": ["Q01", "Q03", "Q04", "Q05", "Q06", "Q07"],
    "urinal_count": ["Q08", "Q09"],
    "pain_count": ["Q10"],
    "miscellaneous_count": ["Q11", "Q28", "Q29"],
    "apathy_attention_memory_count": ["Q12", "Q13", "Q15"],
    "distortion_perception_count": ["Q02", "Q14", "Q30"],
    "depression_anxiety_count": ["Q16", "Q17"],
    "sexual_function_count": ["Q18", "Q19"],
    "cardiovascular_count": ["Q20", "Q21", "Q27"],
    "sleep_fatigue_count": ["Q22", "Q23", "Q24", "Q25", "Q26"]
}

for category, questions in symptom_categories.items():
    questionnaire_clean[category] = (
        questionnaire_clean[questions]
        .sum(axis=1, min_count=1)
    )

questionnaire_clean[
    [
        "patient_id",
        "total_symptom_count",
        *symptom_categories.keys()
    ]
].head()

,patient_id,total_symptom_count,gastrointestinal_count,urinal_count,pain_count,miscellaneous_count,apathy_attention_memory_count,distortion_perception_count,depression_anxiety_count,sexual_function_count,cardiovascular_count,sleep_fatigue_count
0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,12.0,2.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,3.0,3.0
2,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,12.0,1.0,2.0,1.0,1.0,0.0,1.0,1.0,0.0,2.0,3.0
4,5,11.0,2.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,5.0


# Validating the symptom count

In [18]:
print(
    "Minimum symptom count:",
    questionnaire_clean["total_symptom_count"].min()
)

print(
    "Maximum symptom count:",
    questionnaire_clean["total_symptom_count"].max()
)

# Checking whether any values are outside the possible range
invalid_symptom_counts = questionnaire_clean[
    (questionnaire_clean["total_symptom_count"] < 0)
    | (questionnaire_clean["total_symptom_count"] > 30)
]

print(
    "Records with impossible symptom counts:",
    len(invalid_symptom_counts)
)

Minimum symptom count: 0.0
Maximum symptom count: 24.0
Records with impossible symptom counts: 0


# Viewing symptom count summary 

In [19]:
# Summary statistics for total symptoms reported
questionnaire_clean["total_symptom_count"].describe()

count    469.000000
mean       8.057569
std        5.557671
min        0.000000
25%        3.000000
50%        8.000000
75%       12.000000
max       24.000000
Name: total_symptom_count, dtype: float64

# Creating an item level summary 

In [20]:
# Creating a summary for Q01 to Q30
item_summary = pd.DataFrame({
    "question": question_cols,

    "yes_count": [
        questionnaire_clean[col].eq(1).sum()
        for col in question_cols
    ],

    "no_count": [
        questionnaire_clean[col].eq(0).sum()
        for col in question_cols
    ],

    "missing_count": [
        questionnaire_clean[col].isna().sum()
        for col in question_cols
    ]
})

# Calculating the number of valid responses
item_summary["valid_response_count"] = (
    item_summary["yes_count"]
    + item_summary["no_count"]
)

# Calculating the percentage of 1 responses
item_summary["yes_percentage"] = (
    item_summary["yes_count"]
    / item_summary["valid_response_count"]
    * 100
).round(2)

item_summary

,question,yes_count,no_count,missing_count,valid_response_count,yes_percentage
0,Q01,134,335,0,469,28.57
1,Q02,161,308,0,469,34.33
2,Q03,142,327,0,469,30.28
3,Q04,57,412,0,469,12.15
4,Q05,165,304,0,469,35.18
5,Q06,23,446,0,469,4.90
6,Q07,116,353,0,469,24.73
7,Q08,260,209,0,469,55.44
8,Q09,273,196,0,469,58.21
9,Q10,79,390,0,469,16.84


# Sorting questions by reported frequency

In [21]:
# Sorting questions from highest to lowest percentage of 1 responses
item_summary_sorted = item_summary.sort_values(
    by="yes_percentage",
    ascending=False
).reset_index(drop=True)

item_summary_sorted

,question,yes_count,no_count,missing_count,valid_response_count,yes_percentage
0,Q09,273,196,0,469,58.21
1,Q08,260,209,0,469,55.44
2,Q23,216,253,0,469,46.06
3,Q15,194,275,0,469,41.36
4,Q05,165,304,0,469,35.18
5,Q12,165,304,0,469,35.18
6,Q02,161,308,0,469,34.33
7,Q20,157,312,0,469,33.48
8,Q16,154,315,0,469,32.84
9,Q25,153,316,0,469,32.62


# Displaying the five most frequently reported items 

In [22]:
# Displaying the top five questionnaire items
item_summary_sorted.head(5)

,question,yes_count,no_count,missing_count,valid_response_count,yes_percentage
0,Q09,273,196,0,469,58.21
1,Q08,260,209,0,469,55.44
2,Q23,216,253,0,469,46.06
3,Q15,194,275,0,469,41.36
4,Q05,165,304,0,469,35.18


# Creating participant-level summary 

In [23]:
# Creating summary statistics for total symptom count
participant_summary = (
    questionnaire_clean["total_symptom_count"]
    .describe()
    .rename("value")
    .reset_index()
    .rename(columns={"index": "statistic"})
)

participant_summary

,statistic,value
0,count,469.000000
1,mean,8.057569
2,std,5.557671
3,min,0.000000
4,25%,3.000000
5,50%,8.000000
6,75%,12.000000
7,max,24.000000


# Creating questionnaire-complteion summary 

In [24]:
# Counting complete and incomplete questionnaires
completion_summary = (
    questionnaire_clean["questionnaire_status"]
    .value_counts(dropna=False)
    .rename_axis("questionnaire_status")
    .reset_index(name="participant_count")
)

# Calculating percentage
completion_summary["percentage"] = (
    completion_summary["participant_count"]
    / len(questionnaire_clean)
    * 100
).round(2)

completion_summary

,questionnaire_status,participant_count,percentage
0,Complete,469,100.0


# Quality check

In [25]:
print("Final dataset shape:", questionnaire_clean.shape)

print(
    "Remaining duplicate rows:",
    questionnaire_clean.duplicated().sum()
)

print(
    "Missing patient IDs:",
    questionnaire_clean["patient_id"].isna().sum()
)

print(
    "Missing questionnaire responses:",
    questionnaire_clean[question_cols].isna().sum().sum()
)

print(
    "Minimum symptom count:",
    questionnaire_clean["total_symptom_count"].min()
)

print(
    "Maximum symptom count:",
    questionnaire_clean["total_symptom_count"].max()
)

questionnaire_clean.head()

Final dataset shape: (469, 46)
Remaining duplicate rows: 0
Missing patient IDs: 0
Missing questionnaire responses: 0
Minimum symptom count: 0.0
Maximum symptom count: 24.0


,patient_id,questionnaire_name,Q01,Q02,Q03,Q04,Q05,Q06,Q07,Q08,...,gastrointestinal_count,urinal_count,pain_count,miscellaneous_count,apathy_attention_memory_count,distortion_perception_count,depression_anxiety_count,sexual_function_count,cardiovascular_count,sleep_fatigue_count
0,1,NMS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,NMS,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,2.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,3.0,3.0
2,3,NMS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,NMS,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.0,2.0,1.0,1.0,0.0,1.0,1.0,0.0,2.0,3.0
4,5,NMS,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,2.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,5.0


# Saving the cleaned questionnaire

In [26]:
# Setting the output folder
output_folder = Path("../data/processed")

# Creating the folder if it does not exist
output_folder.mkdir(parents=True, exist_ok=True)

# Setting the cleaned file name
cleaned_file = output_folder / "questionnaire_cleaned.csv"

# Saving the cleaned dataset
questionnaire_clean.to_csv(
    cleaned_file,
    index=False
)

print("Cleaned questionnaire saved successfully.")
print("Saved location:", cleaned_file.resolve())

Cleaned questionnaire saved successfully.
Saved location: /Users/user/Documents/GitHub/AI-Assisted-Screening-of-Parkinson-s-Disease/data/processed/questionnaire_cleaned.csv


# Saving the summary files

In [27]:
# Saving item-level summary
item_summary_sorted.to_csv(
    output_folder / "questionnaire_item_summary.csv",
    index=False
)

# Saving participant-level summary
participant_summary.to_csv(
    output_folder / "questionnaire_participant_summary.csv",
    index=False
)

# Saving questionnaire-completion summary
completion_summary.to_csv(
    output_folder / "questionnaire_completion_summary.csv",
    index=False
)

print("All questionnaire summary files were saved successfully.")

All questionnaire summary files were saved successfully.


In [28]:
# Displaying all questionnaire files saved in the processed folder
for file in sorted(output_folder.glob("questionnaire*.csv")):
    print(file.name)

questionnaire_cleaned.csv
questionnaire_completion_summary.csv
questionnaire_item_summary.csv
questionnaire_participant_summary.csv


# Conclusion

## Conclusion

The NMS questionnaire dataset was successfully inspected and cleaned. The responses from Q01 to Q30 were converted into numeric format and checked for missing values, invalid responses, and duplicate records.

Questionnaire-completion variables were created to identify complete and incomplete records. A total symptom-count variable was also generated by adding the binary responses across Q01 to Q30.

Item-level, participant-level, and questionnaire-completion summary tables were created. Finally, the cleaned questionnaire dataset and the summary files were saved in the `data/processed` folder for use in later analysis and machine-learning stages.